# الدرس الرابع: بناء وتوثيق الادوات البرمجية (Implementing Tools)

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعلم كيفية بناء ادوات مخصصة للوكلاء الاذكياء باستخدام الديكوريتور الحديث `@tool` من حزمة `langchain_core.tools`.

## المفاهيم الاساسية لادوات LangChain الحديثة
1. **تحديد الانواع (Type Annotations)**: تستخدم LangChain انواع المدخلات لتوليد مخطط JSON Schema دقيق ترسله الى نموذج اللغة.
2. **التوثيق (Docstrings)**: يعتبر نص التوثيق هو الدليل الوحيد الذي يقرأه النموذج ليقرر متى يستخدم الاداة؛ لذا يجب صياغته بدقة وموضوعية.
3. **مخططات Pydantic للوسائط (Args Schema)**: في حالة الدوال ذات المدخلات المعقدة او القيود الخاصة (Validation rules)، يمكن تمرير نموذج Pydantic مخصص لتأكيد صحة المدخلات قبل تشغيل الدالة.
4. **معالجة الاخطاء (Tool Error Handling)**: معالجة الاستثناءات بأمان واعادة رسائل توضيحية للنموذج لتمكينه من تصحيح اخطائه ذاتيا.

## الخطوة 1: استيراد الاعتماديات وتهيئة البيئة
نقوم باستيراد `@tool` ومكتبات Pydantic وتهيئة النموذج.

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0)

## الخطوة 2: انشاء اداة بسيطة باستخدام الديكوريتور `@tool`
ننشئ اداة حسابية تجري عمليات على الارقام. نلاحظ كيف تستنبط المكتبة مخطط المعاملات تلقائيا من الدالة.

In [ ]:
@tool
def calculate_compound_interest(principal: float, rate: float, time_years: int) -> str:
    """Calculate compound interest total amount and earned interest.
    
    Args:
        principal: The initial invested amount.
        rate: The annual interest rate as a decimal (e.g. 0.05 for 5%).
        time_years: The investment period in years.
    """
    total_amount = principal * ((1 + rate) ** time_years)
    interest_earned = total_amount - principal
    return f"Principal: ${principal:,.2f}, Earned: ${interest_earned:,.2f}, Total: ${total_amount:,.2f}"

# فحص خصائص الاداة البرمجية
print("Tool Name:", calculate_compound_interest.name)
print("Tool Description:", calculate_compound_interest.description)
print("Tool Arguments Schema:", calculate_compound_interest.args)

## الخطوة 3: اداة متقدمة مع مخطط مدخلات مخصص عبر Pydantic
لضمان التحقق الصارم من المعاملات قبل تنفيذ الكود، نستخدم كلاس يرث من `BaseModel` مع استخدام `Field` لوضع قيود وصفية وقيم صغرى/عظمى.

In [ ]:
class SearchEmployeeInput(BaseModel):
    department: str = Field(description="Department name such as Engineering, HR, or Marketing.")
    min_years_experience: int = Field(default=0, ge=0, description="Minimum years of experience, must be non-negative.")

@tool(args_schema=SearchEmployeeInput)
def search_employee_directory(department: str, min_years_experience: int = 0) -> str:
    """Search internal company employee records matching department and experience criteria."""
    mock_db = [
        {"name": "Alice Johnson", "dept": "Engineering", "years": 6},
        {"name": "Bob Smith", "dept": "Engineering", "years": 2},
        {"name": "Carol Danvers", "dept": "Marketing", "years": 5},
    ]
    matches = [
        e["name"] for e in mock_db 
        if e["dept"].lower() == department.lower() and e["years"] >= min_years_experience
    ]
    if not matches:
        return f"No employees found in {department} with at least {min_years_experience} years experience."
    return f"Matching employees in {department}: " + ", ".join(matches)

print("Advanced Tool JSON Schema:")
print(search_employee_directory.args_schema.model_json_schema())

## الخطوة 4: ربط الادوات بالنموذج (Tool Binding) وفحص طلب الاستدعاء
في معمارية LangChain الحديثة، نستخدم `model.bind_tools([tools])` لتمرير مواصفات الادوات الى النموذج عبر واجهة استدعاء الدوال الاصلية (Native Function Calling).

In [ ]:
model_with_tools = model.bind_tools([calculate_compound_interest, search_employee_directory])

# ارسال سؤال يتطلب استخدام الاداة
query = "How much will $10,000 grow into after 5 years at an annual interest rate of 7%?"
ai_msg = model_with_tools.invoke(query)

print("Model Tool Calls Decision:")
print(ai_msg.tool_calls)

## الخطوة 5: تنفيذ الاداة بناء على قرار النموذج
نستخرج بيانات الاستدعاء من `ai_msg.tool_calls` ونمررها للاداة للحصول على النتيجة.

In [ ]:
if ai_msg.tool_calls:
    first_call = ai_msg.tool_calls[0]
    tool_name = first_call["name"]
    tool_args = first_call["args"]
    
    print(f"Executing: {tool_name} with args: {tool_args}")
    if tool_name == "calculate_compound_interest":
        result = calculate_compound_interest.invoke(tool_args)
        print("Tool Output Result:")
        print(result)